In [5]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.9 MB/s eta 0:00:00


In [3]:
import os
import cv2
import shutil
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [1]:
from google.colab import files

uploaded = files.upload()

Saving open video trim2.zip to open video trim2.zip


In [2]:
from google.colab import files

uploaded = files.upload()

Saving open_video_trim_2_landmarks (1).csv to open_video_trim_2_landmarks (1).csv


In [3]:
from google.colab import files

uploaded = files.upload()

Saving standardized_perfect_dataset (1).csv to standardized_perfect_dataset (1).csv


In [4]:
import os

print(os.listdir("/content"))

['.config', 'open video trim2.zip', 'open_video_trim_2_landmarks (1).csv', 'standardized_perfect_dataset (1).csv', 'sample_data']


In [7]:
import zipfile

with zipfile.ZipFile("/content/open video trim2.zip", "r") as zip_ref:
    zip_ref.extractall("/content/open video trim2")

print("Frames Extracted!")

Frames Extracted!


In [8]:
import os

images = sorted(os.listdir("/content/open video trim2"))

print("Total Images:", len(images))
print(images[:5])

Total Images: 332
['frame_0.jpg', 'frame_1.jpg', 'frame_10.jpg', 'frame_100.jpg', 'frame_101.jpg']


In [9]:
import pandas as pd

landmarks = pd.read_csv("/content/open_video_trim_2_landmarks (1).csv")
footwork = pd.read_csv("/content/standardized_perfect_dataset (1).csv")

print("Landmarks:", landmarks.shape)
print("Footwork:", footwork.shape)

Landmarks: (332, 133)
Footwork: (1260, 26)


In [10]:
import zipfile
import os

zip_path = "/content/open video trim2.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/frames")

images = sorted(os.listdir("/content/frames"))

print("Total Images:", len(images))
print(images[:5])

Total Images: 332
['frame_0.jpg', 'frame_1.jpg', 'frame_10.jpg', 'frame_100.jpg', 'frame_101.jpg']


In [11]:
import pandas as pd

landmarks = pd.read_csv("/content/open_video_trim_2_landmarks (1).csv")

footwork = pd.read_csv("/content/standardized_perfect_dataset (1).csv")

print(landmarks.shape)
print(footwork.shape)

(332, 133)
(1260, 26)


In [12]:
import os

IMAGE_WIDTH = 1920
IMAGE_HEIGHT = 1080
PADDING = 20

os.makedirs("/content/labels", exist_ok=True)

for _, row in landmarks.iterrows():

    xs = []
    ys = []

    for i in range(33):

        if row[f"visibility_{i}"] > 0.5:

            xs.append(row[f"x_{i}"] * IMAGE_WIDTH)
            ys.append(row[f"y_{i}"] * IMAGE_HEIGHT)

    xmin = max(min(xs) - PADDING, 0)
    ymin = max(min(ys) - PADDING, 0)

    xmax = min(max(xs) + PADDING, IMAGE_WIDTH)
    ymax = min(max(ys) + PADDING, IMAGE_HEIGHT)

    x = ((xmin + xmax) / 2) / IMAGE_WIDTH
    y = ((ymin + ymax) / 2) / IMAGE_HEIGHT

    w = (xmax - xmin) / IMAGE_WIDTH
    h = (ymax - ymin) / IMAGE_HEIGHT

    label_name = row["frame"].replace(".jpg", ".txt")

    with open(f"/content/labels/{label_name}", "w") as f:
        f.write(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

print("Bounding boxes created:", len(os.listdir("/content/labels")))

Bounding boxes created: 332


In [13]:
video_df = footwork[
    footwork["source_video"] == "open_video_trim_2"
].copy()

print("Frames:", len(video_df))

print("\nClasses:\n")
print(video_df["footwork_label"].value_counts())

Frames: 332

Classes:

footwork_label
Backhand_Backcourt    76
Recovery_Ready        71
Forehand_Backcourt    61
Forehand_Mid          54
Backhand_Mid          47
Backhand_Front        23
Name: count, dtype: int64


In [14]:
classes = sorted(video_df["footwork_label"].unique())

class_map = {cls: idx for idx, cls in enumerate(classes)}

print("Class Mapping:\n")
print(class_map)

Class Mapping:

{'Backhand_Backcourt': 0, 'Backhand_Front': 1, 'Backhand_Mid': 2, 'Forehand_Backcourt': 3, 'Forehand_Mid': 4, 'Recovery_Ready': 5}


In [15]:
video_df["frame"] = video_df["frame"].str.replace("_trim_2", "", regex=False)

video_df.head()

,frame,left_shoulder_x,left_shoulder_y,right_shoulder_x,right_shoulder_y,left_hip_x,left_hip_y,right_hip_x,right_hip_y,left_knee_x,...,right_ankle_y,left_foot_x,left_foot_y,right_foot_x,right_foot_y,source_video,hip_center_x,hip_center_y,stance_width,footwork_label
332,frame_0.jpg,0.400411,0.362875,0.358614,0.368646,0.363918,0.572202,0.340526,0.577582,0.374174,...,0.865539,0.392312,0.865350,0.381110,0.895606,open_video_trim_2,0.352222,0.574892,0.014279,Backhand_Mid
333,frame_1.jpg,0.400188,0.363199,0.357999,0.371150,0.363210,0.561419,0.340236,0.568891,0.369814,...,0.863230,0.392325,0.863952,0.382894,0.894930,open_video_trim_2,0.351723,0.565155,0.012645,Backhand_Mid
334,frame_10.jpg,0.404999,0.365107,0.360702,0.381808,0.363191,0.559393,0.340158,0.567461,0.366684,...,0.861267,0.393628,0.865984,0.383283,0.895012,open_video_trim_2,0.351675,0.563427,0.012301,Backhand_Mid
335,frame_100.jpg,0.449582,0.492437,0.425959,0.441937,0.442022,0.524217,0.419341,0.531870,0.453888,...,0.672839,0.473889,0.665223,0.402412,0.690666,open_video_trim_2,0.430682,0.528044,0.063852,Backhand_Mid
336,frame_101.jpg,0.442292,0.478360,0.476498,0.435311,0.456560,0.477185,0.472478,0.459215,0.443992,...,0.639997,0.446868,0.679270,0.481946,0.654286,open_video_trim_2,0.464519,0.468200,0.026494,Recovery_Ready


In [16]:
import os

os.makedirs("/content/labels_multiclass", exist_ok=True)

count = 0

for _, row in video_df.iterrows():

    frame = row["frame"]
    class_id = class_map[row["footwork_label"]]

    old_label = f"/content/labels/{frame.replace('.jpg', '.txt')}"
    new_label = f"/content/labels_multiclass/{frame.replace('.jpg', '.txt')}"

    if not os.path.exists(old_label):
        continue

    with open(old_label, "r") as f:
        parts = f.read().strip().split()

    # Replace only the class ID
    parts[0] = str(class_id)

    with open(new_label, "w") as f:
        f.write(" ".join(parts))

    count += 1

print(f"✅ Created {count} multiclass labels")

✅ Created 332 multiclass labels


In [17]:
import random

sample = random.choice(os.listdir("/content/labels_multiclass"))

print("Sample file:", sample)

with open(f"/content/labels_multiclass/{sample}") as f:
    print(f.read())

Sample file: frame_241.txt
3 0.714215 0.482140 0.103378 0.403270


In [18]:
import os
import shutil
from sklearn.model_selection import train_test_split

dataset = "/content/badminton_dataset"

folders = [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test"
]

for folder in folders:
    os.makedirs(os.path.join(dataset, folder), exist_ok=True)

images = sorted(os.listdir("/content/frames"))

train_imgs, temp_imgs = train_test_split(
    images,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

val_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_imgs))
print("Validation:", len(val_imgs))
print("Test:", len(test_imgs))

Train: 265
Validation: 33
Test: 34


In [19]:
def copy_dataset(img_list, split):

    for img in img_list:

        shutil.copy(
            f"/content/frames/{img}",
            f"{dataset}/images/{split}/{img}"
        )

        label = img.replace(".jpg", ".txt")

        shutil.copy(
            f"/content/labels_multiclass/{label}",
            f"{dataset}/labels/{split}/{label}"
        )

copy_dataset(train_imgs, "train")
copy_dataset(val_imgs, "val")
copy_dataset(test_imgs, "test")

print("Dataset Ready!")

Dataset Ready!


In [20]:
classes = sorted(video_df["footwork_label"].unique())

with open("/content/dataset.yaml", "w") as f:

    f.write("path: /content/badminton_dataset\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("test: images/test\n\n")

    f.write(f"nc: {len(classes)}\n")

    f.write("names:\n")

    for i, c in enumerate(classes):
        f.write(f"  {i}: {c}\n")

print("dataset.yaml created")

dataset.yaml created


In [21]:
print("Train Images :", len(os.listdir("/content/badminton_dataset/images/train")))
print("Train Labels :", len(os.listdir("/content/badminton_dataset/labels/train")))

print("Val Images :", len(os.listdir("/content/badminton_dataset/images/val")))
print("Val Labels :", len(os.listdir("/content/badminton_dataset/labels/val")))

print("Test Images :", len(os.listdir("/content/badminton_dataset/images/test")))
print("Test Labels :", len(os.listdir("/content/badminton_dataset/labels/test")))

Train Images : 265
Train Labels : 265
Val Images : 33
Val Labels : 33
Test Images : 34
Test Labels : 34


In [22]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="Badminton",
    name="YOLOv8_Multiclass",
    workers=2,
    verbose=True,
    plots=True
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0

In [23]:
metrics = model.val()

print(metrics.results_dict)

Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4039.1±1485.9 MB/s, size: 263.1 KB)
val: Scanning /content/badminton_dataset/labels/val.cache... 33 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 33/33 10.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.2it/s 2.5s
                   all         33         33      0.678      0.593      0.704       0.47
    Backhand_Backcourt          6          6      0.909        0.5      0.696      0.255
        Backhand_Front          4          4      0.851       0.75      0.945      0.892
          Backhand_Mid          6          6      0.416      0.333       0.57      0.352
    Forehand_Backcourt          7          7      0.837      0.738      0.774      0.422
          Forehand_Mid          3          3

In [24]:
from ultralytics import YOLO
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load best model
model = YOLO("/content/Badminton/YOLOv8_Multiclass/weights/best.pt")

# Validate
metrics = model.val(data="/content/dataset.yaml", split="test", plots=True)

# -----------------------------
# Overall Metrics
# -----------------------------
print("="*60)
print("YOLOv8 MULTICLASS RESULTS")
print("="*60)

print(f"Precision      : {metrics.box.mp:.4f}")
print(f"Recall         : {metrics.box.mr:.4f}")
print(f"mAP@0.5        : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95   : {metrics.box.map:.4f}")

# -----------------------------
# Per-Class Results
# -----------------------------
names = metrics.names

report = []

for i, c in names.items():

    report.append({
        "Class": c,
        "AP50-95": metrics.box.maps[i]
    })

report_df = pd.DataFrame(report)

print("\nPer Class AP")
print(report_df)

report_df.to_csv("PerClass_AP.csv", index=False)

print("\nSaved -> PerClass_AP.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Badminton/YOLOv8_Multiclass/weights/best.pt'

In [25]:
import os

for root, dirs, files in os.walk("/content"):
    if "best.pt" in files:
        print(os.path.join(root, "best.pt"))

/content/runs/detect/Badminton/YOLOv8_Multiclass/weights/best.pt


In [26]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/Badminton/YOLOv8_Multiclass/weights/best.pt")

In [27]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/Badminton/YOLOv8_Multiclass/weights/best.pt")

metrics = model.val(
    data="/content/dataset.yaml",
    split="test",
    plots=True
)

Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4385.5±625.9 MB/s, size: 264.2 KB)
val: Scanning /content/badminton_dataset/labels/test... 34 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 34/34 1.1Kit/s 0.0s
val: New cache created: /content/badminton_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.3s/it 3.9s
                   all         34         34      0.368      0.609        0.5      0.368
    Backhand_Backcourt          9          9      0.502      0.556      0.519      0.317
        Backhand_Front          2          2       0.15        0.5      0.495      0.495
          Backhand_Mid          5          5      0.478        0.8      0.735       0.59
    Forehand_Backcourt          2          2     0.0674        0.5     

In [28]:
print(metrics.results_dict)

{'metrics/precision(B)': 0.3676306192480618, 'metrics/recall(B)': 0.6089641043581183, 'metrics/mAP50(B)': 0.5002309611684611, 'metrics/mAP50-95(B)': 0.36773319908952923, 'fitness': 0.36773319908952923}


In [29]:
import pandas as pd

report = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "mAP@0.5",
        "mAP@0.5:0.95"
    ],
    "Value": [
        metrics.box.mp,
        metrics.box.mr,
        metrics.box.map50,
        metrics.box.map
    ]
})

print(report)

report.to_csv("YOLOv8_Report.csv", index=False)

print("✅ Report saved as YOLOv8_Report.csv")

         Metric     Value
0     Precision  0.367631
1        Recall  0.608964
2       mAP@0.5  0.500231
3  mAP@0.5:0.95  0.367733
✅ Report saved as YOLOv8_Report.csv


In [30]:
per_class = pd.DataFrame({
    "Class": list(metrics.names.values()),
    "AP50-95": metrics.box.maps
})

print(per_class)

per_class.to_csv("YOLOv8_PerClass_Report.csv", index=False)

print("✅ Per-class report saved")

                Class   AP50-95
0  Backhand_Backcourt  0.317359
1      Backhand_Front  0.495000
2        Backhand_Mid  0.589925
3  Forehand_Backcourt  0.082562
4        Forehand_Mid  0.512572
5      Recovery_Ready  0.208980
✅ Per-class report saved
